In [1]:
import os, sys, json, math, time, re, random, hashlib, inspect
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List
from functools import lru_cache

import numpy as np
from collections import Counter

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
0 Tesla T4
1 Tesla T4


### Những bổ sung so với baseline 9.82 <nhiều việc quá chưa kịp list, mọi người tự so sánh nhé> 
* Đổi prompt template
* try new training config - t nghĩ mình tìm ra bộ hyperparam tối ưu thì điểm còn tăng nữa
* Training sample theo length




* giảm số chain of thought xuống còn 4 steps
* tăng threshold của arithmetic verification lên 1
* MAX_LENGTH = 256, khi tăng MAX_LENGTH = 384, kết quả giảm còn 28,xx
* Thay đổi normalize response 
* bổ sung function lọc các loại record có vebosity cao 
* Thay đổi COMPUTATION_WEIGHT, ANCHOR_WEIGHT và window khi load dataset




* Sử dụng self consistency - việc tìm top k, top p, temp với num sample tối ưu cũng giúp tăng score đấy, hồi điểm còn thấp nó giúp tăng vài % cơ
* Bỏ repetition penalty, cho vô là điểm thấp hơn idk why



→ Hướng tiếp theo trước khi nộp: t nghĩ hiện tại có thể chốt preprocessing pipeline rồi, mình cần tìm training hyperparam tối ưu, COMPUTATION_WEIGHT, ANCHOR_WEIGHT tối ưu, và infer hyperparam tối ưu 
(!) Thay vì dùng checkpoint cuối, mình nên chọn checkpoint tối ưu á, và bạn t đề xuất là thay đổi lr khi đang train được vài epoch rồi


In [2]:
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
))

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"
TEST_FILE = DATA_DIR / "test.json"

PROMPT_TEMPLATE = "Câu hỏi: {q}\nGiải:\n"
SAFE_EOS_ID = 50256

OUTPUT_DIR = Path("/kaggle/working/gpt2_math_ckpt")
VALID_OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
VALID_REPORT_PATH = Path("/kaggle/working/valid_report.json")
BASELINE_OUTPUT_PATH = Path("/kaggle/working/baseline_valid_output.json")
BASELINE_REPORT_PATH = Path("/kaggle/working/baseline_valid_report.json")
TEST_OUTPUT_PATH = Path("/kaggle/working/test_predictions.json")
EXPERIMENT_REPORT_PATH = Path("/kaggle/working/experiment_report.json")
ERROR_ANALYSIS_PATH = Path("/kaggle/working/error_analysis.json")
HPO_REPORT_PATH = Path("/kaggle/working/hpo_report.json")

# === HPO CONFIG ===
# Edit these between runs. Do not run multiple training experiments in one execution.
FAST_DEV = True
MAX_TRAIN_SAMPLES = 100000 if FAST_DEV else None
MAX_VALID_SAMPLES = 500 if FAST_DEV else None

EPOCHS = 10
LR = 5e-3
WARMUP_RATIO = 0.15
MAX_LENGTH = 256  # Do not change; 384 regressed in prior runs.
MAX_RESPONSE_TOKENS = 256  # Used by the existing conciseness filter.
PER_DEVICE_BATCH_SIZE = 8
GRAD_ACCUM = 4
WEIGHT_DECAY = 0.01
SEED = 42

# Token-level loss weights used by SFTDataset._build_arithmetic_weights.
# Current rewind3 values are exposed here for cross-run manual HPO.
COMPUTATION_WEIGHT = 2.5
ANCHOR_WEIGHT = 6.0
EOS_WEIGHT = 8.0

# Decoding, swept post-training.
MAX_NEW_TOKENS = 80
USE_BEAM_SEARCH = True
NUM_BEAMS = 5
SC_NUM_SAMPLES = 9
SC_TEMPERATURE = 0.4
SC_TOP_K = 40
SC_TOP_P = 0.90

DECODING_GRID = [
    ("beam3",    dict(use_beam=True,  num_beams=3,  sc=False)),
    ("beam5",    dict(use_beam=True,  num_beams=5,  sc=False)),
    ("beam7",    dict(use_beam=True,  num_beams=7,  sc=False)),
    ("sc5_t04",  dict(use_beam=False, sc=True, n=5,  temp=0.4, top_k=40, top_p=0.90)),
    ("sc9_t04",  dict(use_beam=False, sc=True, n=9,  temp=0.4, top_k=40, top_p=0.90)),
    ("sc9_t03",  dict(use_beam=False, sc=True, n=9,  temp=0.3, top_k=30, top_p=0.85)),
    ("sc9_t05",  dict(use_beam=False, sc=True, n=9,  temp=0.5, top_k=50, top_p=0.95)),
    ("sc15_t04", dict(use_beam=False, sc=True, n=15, temp=0.4, top_k=40, top_p=0.90)),
]
TYPE_ROUTING_USED = False
# === END HPO CONFIG ===

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("TRAIN_FILE:", TRAIN_FILE)
print("VALID_FILE:", VALID_FILE)
print("TEST_FILE:", TEST_FILE, "exists=", TEST_FILE.exists())
print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("HPO config:", {
    "epochs": EPOCHS,
    "lr": LR,
    "warmup_ratio": WARMUP_RATIO,
    "max_length": MAX_LENGTH,
    "max_response_tokens": MAX_RESPONSE_TOKENS,
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "computation_weight": COMPUTATION_WEIGHT,
    "anchor_weight": ANCHOR_WEIGHT,
    "max_new_tokens": MAX_NEW_TOKENS,
})


# Training Pipeline

## Evaluation utilities

In [3]:
# ============================================================
# 1.5 Evaluation utilities
# ============================================================
VI_ANCHORS = [
    # 1. SYMBOLIC & PRECISION ANCHORS
    # Catches exact fractions and sqrts safely to prevent regex bleeding
    r"(?i)(?:đáp án|câu trả lời)\s*là\s*[:：]?\s*(\\frac\{[^{}]+\}\{[^{}]+\})",
    r"(?i)(?:đáp án|câu trả lời)\s*là\s*[:：]?\s*(\\sqrt\{[^{}]+\})",
    r"(?i)####\s*đáp\s*án\s*là\s*[:：]?\s*(\\frac\{[^{}]+\}\{[^{}]+\})",
    r"(?i)####\s*đáp\s*án\s*là\s*[:：]?\s*(\\sqrt\{[^{}]+\})",
    
    # 2. STRICT NUMERIC ANCHORS (Avoids capturing trailing text)
    r"(?i)(?:đáp án|câu trả lời)\s*là\s*[:：]?\s*(-?\d+(?:[.,]\d+)?)",
    
    # 3. EXISTING EDA PRECISION ANCHORS (Catch-all to end of line)
    r"(?i)####đáp án là:\s*(.*?)(?=$|\n)",
    r"(?i)(?:đáp án|câu trả lời)\s*là\s*[:：]?\s*(.*?)(?=$|\n)",
    r"####\s*(.*?)(?=$|\n)",
    
    # 4. LEGACY ANCHORS (Fallback)
    r"####đáp án là:\s*",       
    r"####\s*",                 
    r"Đáp án là\s*[:：]?",       
    r"Câu trả lời là\s*[:：]?",  
]

EN_ANCHORS = [
    r"(?i)the answer is\s*[:：]?\s*(-?\d+(?:[.,]\d+)?)",
    r"The answer is\s*[:：]?",
    r"####",
]

BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}

_LATEX_KEEP = re.compile(
    r"(?:-?\d+\\sqrt\{[^{}]+\}|\\sqrt\{[^{}]+\}|-?\d+\\frac\{[^{}]+\}\{[^{}]+\}|\\frac\{[^{}]+\}\{[^{}]+\})"
)


def clean_model_output(text: str) -> str:
    """Strip trailing garbage after ####đáp án là: <value> while preserving supported LaTeX answers."""
    if not text:
        return text

    m = re.search(r"(?i)(####\s*đáp\s*án\s*là\s*[:：]?\s*)", text)
    if not m:
        return text

    prefix = text[:m.end()]
    tail = text[m.end():].lstrip()

    latex_m = _LATEX_KEEP.match(tail)
    if latex_m:
        return prefix + latex_m.group(0)

    num_m = re.match(r"(-?\d+(?:[.,]\d+)?)", tail)
    if num_m:
        return prefix + num_m.group(1)

    return prefix + tail.split()[0] if tail.split() else prefix + tail


def clean_output_records(items: list[dict]) -> list[dict]:
    cleaned = []
    for rec in items:
        out = dict(rec)
        out["model_output"] = clean_model_output(out.get("model_output", ""))
        cleaned.append(out)
    return cleaned


def save_prediction_file(items: list[dict], path: str | Path) -> list[dict]:
    path = Path(path)
    cleaned = clean_output_records(items)
    with path.open("w", encoding="utf-8") as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    print(f"Wrote {path} ({len(cleaned)} records)")
    return cleaned


assert clean_model_output("####đáp án là: 72 ô") == "####đáp án là: 72"
assert clean_model_output("####đáp án là: 5huehue") == "####đáp án là: 5"
assert clean_model_output("####đáp án là: \\frac{1}{2}") == "####đáp án là: \\frac{1}{2}"
assert clean_model_output("####đáp án là: 50\\sqrt{10}") == "####đáp án là: 50\\sqrt{10}"
assert clean_model_output("####đáp án là: 14\\frac{6}{7}") == "####đáp án là: 14\\frac{6}{7}"


def extract_answer(text: str | None, anchors: list[str]) -> str | None:
    """
    Extract final answer by anchors. 
    - If anchor has a capture group (...), returns the exact group (Precision Mode).
    - Otherwise, cuts text after anchor and returns the tail (Legacy Mode).
    - Fallbacks to \boxed{}.
    """
    if not text:
        return None

    for anc in anchors:
        m = re.search(anc, text)
        if m:
            # TÍCH HỢP PRECISION: Kiểm tra xem Regex có capture group () không
            if m.lastindex: 
                return m.group(1).strip() # Lấy chính xác giá trị số trong ngoặc
            
            # LEGACY FALLBACK: Cắt toàn bộ phần đuôi
            tail = text[m.end():].strip()
            tail = tail.split("\n")[0]
            return tail.strip().rstrip(".。、,")

    # Fallback to LaTeX box
    boxes = BOXED_RE.findall(text)
    if boxes:
        return boxes[-1].strip()

    return None


def extract_gold(rec: dict) -> str | None:
    """Gold answer is read only from gold records, not from prediction files."""
    return extract_answer(rec.get("response_vi"), VI_ANCHORS)


def extract_pred(rec: dict) -> str | None:
    """
    Version 1 with defensive numeric cleanup.
    Works correctly when truncate_at_answer() is applied upstream.
    """
    text = rec.get("model_output", "")
    if not text:
        return None

    # Primary: precision anchor extraction
    ans = extract_answer(text, VI_ANCHORS + EN_ANCHORS)

    if ans is not None:
        # Defensive cleanup: if legacy anchor fired and returned tail text,
        # extract just the first number from it
        # If precision anchor fired, ans is already a pure number — this is a no-op
        num_match = re.search(r'-?\d+(?:[.,]\d+)?', ans)
        if num_match:
            return num_match.group(0)
        # Anchor found but no number — genuine failure
        return None

    # Last resort: the anchor was somehow absent (should not happen after truncation)
    # Return last number in text rather than None — score=1 is better than score=0
    nums = re.findall(r'-?\d+(?:[.,]\d+)?', text)
    return nums[-1] if nums else None
    

def parse_number(s: str | None) -> float | None:
    """Best-effort parser for finite scalar numeric answers."""
    if s is None:
        return None

    t = s.strip()
    if not t:
        return None

    # Pure Vietnamese decimal: 1,5 -> 1.5
    if re.fullmatch(r"-?\d+,\d+", t):
        try:
            return float(t.replace(",", "."))
        except ValueError:
            return None

    # Pure English decimal / integer
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", t):
        try:
            val = float(t)
            return val if math.isfinite(val) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")

    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

# ── AST Compiler Protection (Fix for Float SyntaxWarning) ──────────
    # Implicit multiplication hardened for decimals and bracket-to-bracket
    t = re.sub(r"([\d.])\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|[\d.])", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|[\d.]|\()", r"\1*\2", t)
    t = re.sub(r"\)\s*\(", r")*(", t) # <--- CRITICAL FIX FOR (1.5)(2.5)
    # ───────────────────────────────────────────────────────────────────

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")

    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)

    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    t = t.replace("^", "**")

    try:
        val = eval(t, {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(val, bool):
        return None

    if isinstance(val, (int, float)):
        val = float(val)
        return val if math.isfinite(val) else None

    return None

def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None

    denom = max(1.0, abs(gold))
    return abs(pred - gold) / denom

def score_one(re_val: float | None, extractable: bool) -> int:
    if not extractable:
        return 0
    if re_val is None:
        return 0
    if re_val <= 0.01:
        return 10
    if re_val <= 0.10:
        return 5
    if re_val <= 0.50:
        return 1
    return 0

def align_predictions_with_gold(pred_items: list[dict], gold_items: list[dict]) -> list[tuple[dict, dict]]:
    pred_has_id = all("id" in x for x in pred_items)
    gold_has_id = all("id" in x for x in gold_items)

    if pred_has_id and gold_has_id:
        pred_map = {str(x["id"]): x for x in pred_items}
        pairs = []

        missing = []
        for g in gold_items:
            gid = str(g["id"])
            if gid not in pred_map:
                missing.append(gid)
            else:
                pairs.append((pred_map[gid], g))

        if missing:
            raise ValueError(f"Prediction thiếu {len(missing)} id, ví dụ: {missing[:5]}")

        return pairs

    if len(pred_items) != len(gold_items):
        raise ValueError(
            f"Số lượng prediction ({len(pred_items)}) khác số lượng gold ({len(gold_items)})."
        )

    return list(zip(pred_items, gold_items))

def compute_rel_error_diagnostics(rows: list[dict]) -> dict:
    """
    rows: list of score-annotated dicts with keys 'gold_num', 'pred_num', 'score'.
    Returns extended relative-error diagnostics.
    """
    errors = []
    for r in rows:
        g = r.get("gold_num")
        p = r.get("pred_num")
        if g is None or p is None:
            continue
        try:
            denom = max(1.0, abs(float(g)))
            errors.append(abs(float(p) - float(g)) / denom)
        except (ValueError, TypeError):
            continue

    if not errors:
        return {}

    errors_np = np.array(errors)
    cap = np.median(errors_np) * 10

    return {
        "n_with_error": len(errors),
        "mean_rel_error": float(np.mean(errors_np)),
        "median_rel_error": float(np.median(errors_np)),
        "p90_rel_error": float(np.percentile(errors_np, 90)),
        "p95_rel_error": float(np.percentile(errors_np, 95)),
        "p99_rel_error": float(np.percentile(errors_np, 99)),
        "capped_mean_rel_error": float(np.mean(np.minimum(errors_np, cap))),
        "worst_5_errors": [float(x) for x in sorted(errors, reverse=True)[:5]],
    }


def compute_score_by_type(rows: list[dict]) -> dict:
    grouped: dict[str, dict] = {}
    for r in rows:
        t = r.get("type") or "UNKNOWN"
        g = grouped.setdefault(t, {
            "n": 0,
            "raw_score": 0,
            "exact_count": 0,
            "extractable": 0,
            "numeric_pairs": 0,
            "buckets": {"10": 0, "5": 0, "1": 0, "0": 0},
        })
        score = int(r.get("score", 0))
        g["n"] += 1
        g["raw_score"] += score
        g["exact_count"] += int(score == 10)
        g["extractable"] += int(bool(r.get("extractable")))
        g["numeric_pairs"] += int(r.get("gold_num") is not None and r.get("pred_num") is not None)
        g["buckets"][str(score)] = g["buckets"].get(str(score), 0) + 1

    for g in grouped.values():
        n = max(1, g["n"])
        g["mean_score"] = g["raw_score"] / n
        g["score_pct"] = g["raw_score"] / (10 * n)
        g["extractable_rate"] = g["extractable"] / n
        g["numeric_pair_rate"] = g["numeric_pairs"] / n
    return grouped


def evaluate(pred_items: list[dict], gold_items: list[dict]) -> dict:
    pairs = align_predictions_with_gold(pred_items, gold_items)

    rows = []
    total = 0
    bucket10 = bucket5 = bucket1 = bucket0 = 0
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in pairs:
        gold_ans = extract_gold(gold_rec)
        pred_ans = extract_pred(pred_rec)

        is_extractable = pred_ans is not None
        extractable += int(is_extractable)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)
        re_val = rel_error(pred_num, gold_num)

        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s

        bucket10 += int(s == 10)
        bucket5 += int(s == 5)
        bucket1 += int(s == 1)
        bucket0 += int(s == 0)

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    max_score = n * 10
    score_10 = total / n if n else 0.0
    diagnostics = compute_rel_error_diagnostics(rows)
    score_by_type = compute_score_by_type(rows)

    summary = {
        "n": n,
        "raw_score": total,
        "max_raw_score": max_score,
        "score_10": score_10,
        "score_pct": total / max_score if max_score else 0.0,
        "extractable": extractable,
        "extractable_count": extractable,
        "numeric_pairs": numeric_pairs,
        "numeric_pair_count": numeric_pairs,
        "exact_count": bucket10,
        "buckets": {
            "10": bucket10,
            "5": bucket5,
            "1": bucket1,
            "0": bucket0,
        },
        "bucket_distribution": {
            "10": bucket10,
            "5": bucket5,
            "1": bucket1,
            "0": bucket0,
        },
        "score_by_type": score_by_type,
        "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        "rel_error_diagnostics": diagnostics,
    }
    summary.update(diagnostics)

    return {
        "summary": summary,
        "rows": rows,
    }


def save_evaluation_report(
    pred_path: str | Path,
    gold_records: list[dict],
    report_path: str | Path,
) -> dict:
    pred_path = Path(pred_path)
    report_path = Path(report_path)

    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)

    result = evaluate(pred_items, gold_records)

    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Wrote {report_path}")
    return result

## Data loading & Preprocessing

### Normalize Vietnamese Word Number

Manh thấy có những record thế này: Alice di chuyển lên trên ba ô, sang trái tám ô,... rồi hỏi cuối cùng cần di chuyển bao nhiêu ô. Nên manh nghĩ là nên thay các word numbers thành number và các từ chỉ phép toán cộng trừ nhân chia cũng chuyển thành operator luôn.

In [4]:
# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — Core lookup tables
# ══════════════════════════════════════════════════════════════════════════
VIET_DIGIT_MAP: dict[str, int] = {
    "không": 0,
    "một": 1,
    "hai": 2,
    "ba": 3,
    "bốn": 4,
    "năm": 5,
    "sáu": 6,
    "bảy": 7,
    "tám": 8,
    "chín": 9,
    "mười": 10,
}
_SIMPLE: dict[str, int] = dict(VIET_DIGIT_MAP)
assert "bay" not in _SIMPLE, "Do not normalize the common Vietnamese word 'bay' as digit 7."

_MAG: dict[str, int] = {
    "mươi":  10,
    "mười":  10,
    "trăm":  100,
    "nghìn": 1_000, "ngàn": 1_000,
    "triệu": 1_000_000,
    "tỷ":    1_000_000_000, "tỉ": 1_000_000_000,
}

_ALL_WORDS: list[str] = sorted(
    list(_SIMPLE.keys()) + list(_MAG.keys()),
    key=len, reverse=True
)

_NUM_WORD_ALT: str = "|".join(re.escape(w) for w in _ALL_WORDS)
_DIGIT_OR_WORD: str = rf"(?:\d+(?:[.,]\d+)?|{_NUM_WORD_ALT})"


# ══════════════════════════════════════════════════════════════════════════
# SECTION 2 — Guard tables
# ══════════════════════════════════════════════════════════════════════════

# FIX-4: added "lần" (ordinal counter: "lần đầu" = first time)
_ORDINAL_GUARDS: frozenset[str] = frozenset([
    "thứ",    # thứ hai = second, thứ ba = third
    "tháng",  # tháng một = January
    "ngày",   # ngày một = day 1 (calendar)
    "tuần",   # tuần một = week 1
    "lần",    # lần một = first time  ← NEW
])

# FIX-1: temporal unit set — if previous output token ends with a digit,
# these words denote duration/time, NOT a numeric value.
_TEMPORAL_UNITS: frozenset[str] = frozenset([
    "năm",    # years  (also = 5, hence the ambiguity)
    "tháng",  # months (also guarded above as ordinal)
    "tuần",   # weeks
    "ngày",   # days
    "giờ",    # hours
    "phút",   # minutes
    "giây",   # seconds
])

# FIX-2: semantic pairs where a classifier word precedes a number word
# and the combination should NOT be digit-converted.
# { prefix_word: {protected_number_words} }
_SEMANTIC_PROTECTED_PAIRS: dict[str, frozenset[str]] = {
    "bậc":  frozenset(["một", "hai", "ba", "bốn", "năm"]),   # bậc hai = squared
    "loại": frozenset(["một", "hai", "ba", "bốn"]),           # loại một = category 1
    "hạng": frozenset(["một", "hai", "ba", "bốn"]),  
    "phần": frozenset(["trăm"]),                             # FIX: Protect phần trăm# hạng nhất/hai
}


# ══════════════════════════════════════════════════════════════════════════
# SECTION 3 — Compound number parser  (unchanged from original)
# ══════════════════════════════════════════════════════════════════════════

def _parse_compound_number(tokens: list[str]) -> tuple[int | None, int]:
    """
    Greedily parse a Vietnamese compound number from the token list.
    Returns (value, tokens_consumed) or (None, 0).
    """
    if not tokens:
        return None, 0

    t = tokens[0].lower()
    if t not in _SIMPLE and t not in _MAG:
        return None, 0

    if t in _MAG and t not in _SIMPLE:
        mag = _MAG[t]
        consumed = 1
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < mag:
            return mag + sub_val, consumed + sub_consumed
        return mag, consumed

    lead = _SIMPLE[t]
    consumed = 1

    if consumed >= len(tokens):
        return lead, consumed

    nxt = tokens[consumed].lower()

    if nxt in ("mươi", "mười") and lead >= 2:
        consumed += 1
        base = lead * 10
        if consumed < len(tokens):
            nnxt = tokens[consumed].lower()
            if nnxt in _SIMPLE and nnxt not in ("mươi", "mười"):
                base += _SIMPLE[nnxt]
                consumed += 1
        return base, consumed

    if nxt == "mười" and lead == 1:
        consumed += 1
        base = 10
        if consumed < len(tokens):
            nnxt = tokens[consumed].lower()
            if nnxt in _SIMPLE and nnxt not in ("mươi", "mười"):
                base += _SIMPLE[nnxt]
                consumed += 1
        return base, consumed

    if t in ("mười", "muời") and nxt in _SIMPLE and nxt not in ("mươi", "mười"):
        return 10 + _SIMPLE[nxt], 2

    if nxt == "trăm":
        consumed += 1
        base = lead * 100
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < 100:
            return base + sub_val, consumed + sub_consumed
        return base, consumed

    if nxt in ("nghìn", "ngàn"):
        consumed += 1
        base = lead * 1_000
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < 1_000:
            return base + sub_val, consumed + sub_consumed
        return base, consumed

    if nxt == "triệu":
        consumed += 1
        base = lead * 1_000_000
        sub_val, sub_consumed = _parse_compound_number(tokens[consumed:])
        if sub_val is not None and sub_val < 1_000_000:
            return base + sub_val, consumed + sub_consumed
        return base, consumed

    if nxt in ("tỷ", "tỉ"):
        consumed += 1
        return lead * 1_000_000_000, consumed

    return lead, 1


# ══════════════════════════════════════════════════════════════════════════
# SECTION 4 — Public function 1: number-word normalizer
# ══════════════════════════════════════════════════════════════════════════

def normalize_vietnamese_number_words(text: str) -> str:
    if not isinstance(text, str):
        return text

    tokens = text.split()
    out: list[str] = []
    i = 0

    while i < len(tokens):
        tok_lower = tokens[i].lower()

        # ── Ordinal / calendar guard ──────────────────────────────────
        if tok_lower in _ORDINAL_GUARDS:
            out.append(tokens[i])
            i += 1
            if i < len(tokens) and tokens[i].lower() in _SIMPLE:
                out.append(tokens[i])
                i += 1
            continue
            
        if tok_lower in _TEMPORAL_UNITS:
            prev_out = out[-1] if out else ""
            if re.search(r"\d$", prev_out):
                # Previous token ended with a digit → this is a duration word
                out.append(tokens[i])
                i += 1
                continue
      
        if tok_lower in _SIMPLE or tok_lower in _MAG:
            prev_out = out[-1].lower().strip(".,;:!?") if out else ""
            if (prev_out in _SEMANTIC_PROTECTED_PAIRS and
                    tok_lower in _SEMANTIC_PROTECTED_PAIRS[prev_out]):
                out.append(tokens[i])
                i += 1
                continue
        
        if tok_lower == "nửa":
            prev_is_num = bool(out and re.search(r"\d", out[-1]))
            next_is_num = (i + 1 < len(tokens) and
                           re.search(r"\d|[+\-*/]", tokens[i + 1]))
            out.append("0.5" if (prev_is_num or next_is_num) else tokens[i])
            i += 1
            continue

        if tok_lower in _SIMPLE or tok_lower in _MAG:
            val, consumed = _parse_compound_number(tokens[i:])
            if val is not None and consumed > 0:
                out.append(str(val))
                i += consumed
                continue
                
        out.append(tokens[i])
        i += 1

    return " ".join(out)


# ══════════════════════════════════════════════════════════════════════════
# SECTION 5 — Public function 2: math-operator normalizer
# ══════════════════════════════════════════════════════════════════════════

_NB = rf"(?:{_DIGIT_OR_WORD})"

_NHAN_NON_ARITH_FOLLOW: re.Pattern = re.compile(
    r"^(viên|vật|dân|loại|tố|lực|chứng|quyền|giống|tạo|bản|từ|điển|ái|đức|nghĩa|hậu)\b",
    re.IGNORECASE,
)

_CHIA_NON_ARITH_FOLLOW: re.Pattern = re.compile(
    r"^(sẻ|tay|rẽ|cắt|lửa|buồn|vui|ngọt)\b",
    re.IGNORECASE,
)

_FRACTION_RE: re.Pattern = re.compile(
    rf"({_NB})\s+phần\s+({_NB})",
    re.IGNORECASE,
)

_PCT_RE: re.Pattern = re.compile(
    rf"({_NB})\s+phần\s+trăm",
    re.IGNORECASE,
)

_STANDALONE_FRAC_DIGIT_RE: re.Pattern = re.compile(
    r"\bphần\s+(\d+)\b"
)

_NHAN_VOI_RE: re.Pattern = re.compile(r"(?<=\d\s)nhân\s+với(?=\s)", re.IGNORECASE)
_CHIA_CHO_RE: re.Pattern = re.compile(r"(?<=\d\s)chia\s+cho(?=\s)", re.IGNORECASE)
_CONG_VOI_RE: re.Pattern = re.compile(r"(?<=\d\s)cộng\s+với(?=\s)", re.IGNORECASE)
_TRU_DI_RE:  re.Pattern = re.compile(r"(?<=\d\s)trừ\s+đi(?=\s)", re.IGNORECASE)

_GAP_RE: re.Pattern = re.compile(
    r"\bgấp\s+(\d+(?:[.,]\d+)?|" + _NUM_WORD_ALT + r")(?:\s+lần)?",
    re.IGNORECASE,
)

def _is_fraction_context(m: re.Match, text: str) -> bool:
    numerator_str = m.group(1)
    return bool(re.search(r"\d", numerator_str) or
                numerator_str.lower() in _SIMPLE)


def normalize_vietnamese_math_operators(text: str, *,
                                        convert_pct: bool = True,
                                        convert_gap: bool = True) -> str:
    if not isinstance(text, str):
        return text

    # Step 0: word→digit (idempotent)
    text = normalize_vietnamese_number_words(text)

    # Step 1: percent  "N phần trăm" → "N%"
    if convert_pct:
        text = _PCT_RE.sub(lambda m: m.group(1) + "%", text)

    # Step 2 MUST run before Step 3 to prevent "1 phần 3" from turning into "1 1/3".
    prev = None
    while prev != text:
        prev = text
        text = _FRACTION_RE.sub(
            lambda m: (f"{m.group(1)}/{m.group(2)}"
                       if _is_fraction_context(m, text)
                       else m.group(0)),
            text,
        )

    # Step 3: standalone fraction  "phần 3" → "1/3"
    text = _STANDALONE_FRAC_DIGIT_RE.sub(lambda m: f"1/{m.group(1)}", text)

    # Step 4: verb-phrase operators (unambiguous multi-word)
    text = _NHAN_VOI_RE.sub("*", text)
    text = _CHIA_CHO_RE.sub("/", text)
    text = _CONG_VOI_RE.sub("+", text)
    text = _TRU_DI_RE.sub("-", text)

    #Step 5: "gấp N [lần]" → "* N"
    if convert_gap:
        def _gap_repl(m: re.Match) -> str:
            raw_n = m.group(1)
            digit_n = normalize_vietnamese_number_words(raw_n)
            return f"* {digit_n}"
        text = _GAP_RE.sub(_gap_repl, text)

    # Step 6: context-guarded single-word operators
    tokens = text.split(" ")
    out: list[str] = []
    
    for idx, tok in enumerate(tokens):
        tok_l = tok.lower()

        if tok_l == "cộng":
            if _numeric_neighbour(tokens, idx):
                out.append("+"); continue

        elif tok_l == "trừ":
            next_tok = tokens[idx + 1].lower() if idx + 1 < len(tokens) else ""
            if next_tok not in ("phi", "khi") and _numeric_neighbour(tokens, idx):
                out.append("-"); continue

        elif tok_l == "nhân":
            next_tok = tokens[idx + 1] if idx + 1 < len(tokens) else ""
            if not _NHAN_NON_ARITH_FOLLOW.match(next_tok) and _numeric_neighbour(tokens, idx):
                out.append("*"); continue

        elif tok_l == "chia":
            next_tok = tokens[idx + 1] if idx + 1 < len(tokens) else ""
            if not _CHIA_NON_ARITH_FOLLOW.match(next_tok) and _numeric_neighbour(tokens, idx):
                out.append("/"); continue

        out.append(tok)

    return " ".join(out)

def _numeric_neighbour(tokens: list[str], idx: int, window: int = 1) -> bool:
    _num_word_set = set(_SIMPLE.keys()) | set(_MAG.keys())
    for delta in range(1, window + 2):
        for sign in (-1, 1):
            j = idx + sign * delta
            if 0 <= j < len(tokens):
                t = tokens[j].lower().strip(".,;:!?()")
                if re.search(r"\d", t) or t in _num_word_set:
                    return True
    return False

# ══════════════════════════════════════════════════════════════════════════
# SECTION 6 — Combined convenience wrapper
# ══════════════════════════════════════════════════════════════════════════
import unicodedata

def normalize_vi_math_text(text: str) -> str:
    """
    Apply both normalisers in the correct order.
    Call this BEFORE clean_math_text() in the training pipeline.
    """
    # Normalize to NFC before any Vietnamese word matching
    text = unicodedata.normalize('NFC', text)
    text = normalize_vietnamese_number_words(text)
    text = normalize_vietnamese_math_operators(text)
    return text

In [11]:
# ============================================================
# 2. Data loading & Preprocessing
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        records = json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]
    # Normalize all string fields to NFC at load time
    for rec in records:
        for key in rec:
            if isinstance(rec[key], str):
                rec[key] = unicodedata.normalize('NFC', rec[key])
    return records
    
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

# ============================================================
# Text & Typographical Cleansing
# ============================================================
def clean_math_text(text: str) -> str:
    if not isinstance(text, str): return text

    # Protect <<expr=result>> annotations BEFORE any substitutions
    # Extract and temporarily replace them
    placeholders = {}
    def protect_annotation(m):
        key = f"__ANNOT_{len(placeholders)}__"
        placeholders[key] = m.group(0)
        return key
    
    text = re.sub(r'<<[^>]+>>', protect_annotation, text)
    
    # 1. LaTeX cleanup
    text = re.sub(r'\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'(\1/\2)', text)
    text = re.sub(r'\\sqrt\s*\{([^{}]+)\}', r'sqrt(\1)', text)
    text = text.replace('\\times', ' * ').replace('\\cdot', ' * ')
    text = re.sub(r'\\angle\s*([A-Z]{2,3})', r'góc \1', text)
    text = re.sub(r'\\triangle\s*([A-Z]{2,3})', r'tam giác \1', text)
    text = re.sub(r'\\[a-zA-Z]+\{([^{}]*)\}', r'\1', text)
    text = re.sub(r'\\[a-zA-Z]+\s', ' ', text)
    text = re.sub(r'\$([^$]{1,100})\$', r'\1', text)
    text = re.sub(r'\$\$([^$]+)\$\$', r'\1', text)
    
    # 2. Fix digit * digit corruptions
    text = re.sub(r'((?<=\d)\s*_\s*(?=\d))', ' * ', text)
    text = re.sub(r'([A-Za-z])_(\d+)', r'\1\2', text)
    text = re.sub(r'\^\{(\d+)\}', r'^\1', text)
    
    # 3. Numeric normalizations
    text = re.sub(r'(?<=\d),(?=\d{3}\b)', '', text)
    text = re.sub(r'(?<=\d)\.(?=\d{3}\b)', '', text)
    text = re.sub(r'(?<=\d),(?=\d)', '.', text)
    
    # Restore protected annotations
    for key, val in placeholders.items():
        text = text.replace(key, val)
    
    return re.sub(r'\s+', ' ', text).strip()

# ============================================================
# Format Normalization & Strict Numeric Sanitization
# ============================================================
def extract_computation_chain(response_vi: str) -> str | None:
    """
    Extract only lines containing verifiable arithmetic from a Vietnamese response.
    Returns a compact computation chain or None if fewer than 1 valid equation found.
    """
    lines = response_vi.split(".")
    chain_parts = []
    
    # Pattern: lines with at least one arithmetic expression "X op Y = Z"
    eq_pattern = re.compile(
        r'[-\d\s\(\)\+\-\*\/\.,]+'    # expression side
        r'\s*=\s*'
        r'-?\d+(?:[.,]\d+)?'           # result side
    )
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if eq_pattern.search(line):
            # Keep only the mathematical part, strip leading Vietnamese prose
            # Find where the first number/operator starts
            math_start = re.search(r'\d', line)
            if math_start:
                chain_parts.append(line[math_start.start():].strip())
    
    return ". ".join(chain_parts) if chain_parts else None


def normalize_response(response: str) -> str | None:
    """
    Template A compatible: compact computation chain format.
    """
    # 1. Extract the gold numeric answer
    gold_str = None
    for pat in [
        r"(?i)(?:đáp án|câu trả lời)\s*là\s*[:：]?\s*([^\n]+)",
        r"####\s*([^\n]+)"
    ]:
        m = re.search(pat, response)
        if m:
            gold_str = m.group(1)
            break
    
    if not gold_str:
        return None
    
    gold_num = parse_number(gold_str.replace(',', '.'))
    if gold_num is None or isinstance(gold_num, bool):
        return None
    
    # 2. Build compact chain
    chain = extract_computation_chain(response)
    
    clean_num = str(int(gold_num)) if gold_num == int(gold_num) else str(gold_num)
    
    if chain:
        # Template A format — NO <|endoftext|> string (SFTDataset appends real EOS token)
        return f"{chain}\n####đáp án là: {clean_num}"
    else:
        # Fallback: no chain extractable, just emit the answer anchor
        # Still better than None — at least the answer token trains
        return f"####đáp án là: {clean_num}"

def is_short_cot(rec: dict, max_steps: int = 5) -> bool:
    """
    Giữ lại các sample có độ dài chuỗi suy luận (CoT) <= max_steps.
    """
    target = rec.get('_target', '')
    step_pattern = re.compile(r'[\d\.\,]+\s*[\+\-\*\/]\s*[\d\s\+\-\*\/\(\)\.\,]*\s*=\s*-?[\d\.\,]+')
    steps = step_pattern.findall(target)
    # Loại bỏ các bài không có phép tính (<= 0) và quá dài (> 4)
    return len(steps) <= max_steps

def is_concise_response(rec: dict, tokenizer, max_response_tokens: int = 256) -> bool:
    """
    Threshold raised to 256 to allow Patch 5 (diversity retention) to survive.
    Filters out extreme outliers that will bloat VRAM.
    """
    target = rec.get('_target', '')
    r_ids = tokenizer(target, add_special_tokens=False)['input_ids']
    return len(r_ids) <= max_response_tokens
    

def compute_arithmetic_score(response_text: str) -> float:
    """
    Returns:
        1.0  — no verifiable arithmetic found (neutral, don't penalize)
        0.0–1.0 — fraction of verifiable expressions that are correct
    """
    response_text = clean_math_text(response_text)
    
    # Strict pattern: ONLY pure numeric expressions, no letters allowed nearby
    strict_pattern = re.compile(
        r'(?<![A-Za-zÀ-ỹ])'           # Not preceded by letter
        r'([\d\s\+\-\*\/\(\)\.]{3,})'  # Expression: at least 3 chars
        r'\s*=\s*'
        r'(-?[\d\.]+)'                  # Result
        r'(?![A-Za-zÀ-ỹ\d])'          # Not followed by letter or digit
    )
    matches = strict_pattern.findall(response_text)
    
    if not matches:
        return 1.0  # Neutral

    correct = total = 0
    for expr_str, result_str in matches:
        expr_str = expr_str.strip()
        # Final guard: reject if any letters slipped through
        if re.search(r'[A-Za-zÀ-ỹ]', expr_str):
            continue
        # Reject trivial "X = X" (just an assignment, not a computation)
        if re.fullmatch(r'\s*[\d\.]+\s*', expr_str):
            continue
        expr_str = re.sub(r'([\d.])\s*\(', r'\1*(', expr_str)
        expr_str = re.sub(r'\)\s*([\d.])', r')*\1', expr_str)
        expr_str = re.sub(r'\)\s*\(', r')*(', expr_str)
        try:
            computed = eval(expr_str, {"__builtins__": {}}, {})
            expected = float(result_str.strip())
            denom = max(1.0, abs(expected))
            if abs(float(computed) - expected) / denom < 0.02:
                correct += 1
            total += 1
        except Exception:
            pass  # Unevaluable → skip, don't penalize

    return (correct / total) if total > 0 else 1.0

def is_arithmetically_verified(rec: dict, threshold: float = 1.0) -> bool:
    """
    Filter: only keep training samples where at least 75% of 
    verifiable arithmetic statements in the response are correct.
    """
    return compute_arithmetic_score(rec.get('_target', '')) >= threshold

def is_final_answer_correct(rec: dict) -> bool:
    """Cross-validate: Đảm bảo output text thực sự chứa số khớp với gold."""
    gold_str = extract_answer(rec.get('response_vi', ''), VI_ANCHORS)
    parsed_num = parse_number(gold_str)
    return parsed_num is not None

def is_valid_length(rec, tokenizer, max_total_length=MAX_LENGTH, min_resp_len=20):
    prompt = PROMPT_TEMPLATE.format(q=rec['query_vi'].strip())
    p_len = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
    r_len = len(tokenizer(rec['_target'].strip(), add_special_tokens=False)['input_ids'])
    
    if (p_len + r_len + 1) > max_total_length:
        return False
    if r_len < min_resp_len:
        return False
    return True
    
def is_extractable_target(target_text: str) -> bool:
    extracted_str = extract_answer(target_text, VI_ANCHORS)
    
    if extracted_str is None:
        return False
        
    parsed_val = parse_number(extracted_str)
    
    # Nếu parse ra None, hoặc ra Boolean (lỗi logic), loại bỏ.
    if parsed_val is None or isinstance(parsed_val, bool):
        return False
        
    return True

In [6]:
raw_train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)

**Manh xem phần lớn records có len bao nhiêu để set MAX_LENGTH á Điệp**

In [7]:
import numpy as np
import matplotlib.pyplot as plt

def analyze_token_lengths(records, tokenizer, prompt_template):
    lengths = []
    
    for rec in records:
        # Lấy text thô như cách bạn đưa vào model
        prompt = prompt_template.format(q=rec['query_vi'].strip())
        target = rec.get('_target', rec.get('response_vi', '')).strip()
        
        # Mã hóa thành tokens (không tự động thêm special tokens để đếm chính xác)
        p_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
        r_ids = tokenizer(target, add_special_tokens=False)['input_ids']
        
        # Tổng chiều dài: Prompt + Target + 1 (SAFE_EOS_ID)
        total_len = len(p_ids) + len(r_ids) + 1
        lengths.append(total_len)

    lengths = np.array(lengths)
    
    print("="*50)
    print(" BÁO CÁO EDA CHIỀU DÀI SEQUENCE (TOKENS)")
    print("="*50)
    print(f"Tổng số samples: {len(lengths):,}")
    print(f"Mean:   {np.mean(lengths):.1f}")
    print(f"Median: {np.median(lengths):.1f}")
    print(f"Max:    {np.max(lengths)}")
    print("-"*50)
    
    # Tính các mốc Percentile quan trọng
    percentiles = [50, 75, 85, 90, 95, 98, 99, 99.5, 100]
    for p in percentiles:
        val = np.percentile(lengths, p)
        print(f"Percentile {p:4.1f}% bao phủ ngưỡng: {val:.0f} tokens")
        
    return lengths

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)

In [8]:
seq_lengths = analyze_token_lengths(valid_records, tokenizer, PROMPT_TEMPLATE)

 BÁO CÁO EDA CHIỀU DÀI SEQUENCE (TOKENS)
Tổng số samples: 1,000
Mean:   238.6
Median: 208.0
Max:    1003
--------------------------------------------------
Percentile 50.0% bao phủ ngưỡng: 208 tokens
Percentile 75.0% bao phủ ngưỡng: 296 tokens
Percentile 85.0% bao phủ ngưỡng: 356 tokens
Percentile 90.0% bao phủ ngưỡng: 392 tokens
Percentile 95.0% bao phủ ngưỡng: 488 tokens
Percentile 98.0% bao phủ ngưỡng: 587 tokens
Percentile 99.0% bao phủ ngưỡng: 667 tokens
Percentile 99.5% bao phủ ngưỡng: 805 tokens
Percentile 100.0% bao phủ ngưỡng: 1003 tokens


**Chỗ này manh lọc record theo loại câu hỏi toán**

In [9]:
import random

print(f"Loading tokenizer for token-aware length gating (Patch 3 & 5)...")
temp_tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
temp_tok.pad_token_id = SAFE_EOS_ID 
temp_tok.eos_token_id = SAFE_EOS_ID

print(f"Starting pipeline with {len(raw_train_records)} raw train records.")

allowed_unconditionally = {"GSM_AnsAug", "GSM_Rephrased", "MATH_AnsAug", "MATH_Rephrased"}
verbose_tracks = {"GSM_SV", "GSM_FOBAR", "MATH_SV", "MATH_FOBAR"}

MAX_VERBOSE_TOKENS = 100
VERBOSE_KEEP_PROB = 0.15

filtered_train_records = []
sv_fobar_kept_short = 0
sv_fobar_kept_verbose = 0
sv_fobar_dropped = 0

for r in raw_train_records:
    rtype = r.get("type")
    
    if rtype in allowed_unconditionally:
        filtered_train_records.append(r)
    elif rtype in verbose_tracks:
        # PATCH 3: Token-aware length gating
        response_text = r.get("response_vi", "")
        tok_len = len(temp_tok(response_text, add_special_tokens=False)['input_ids'])
        
        if tok_len <= MAX_VERBOSE_TOKENS: 
            filtered_train_records.append(r)
            sv_fobar_kept_short += 1
        else:
            # PATCH 5: Anti-mode-collapse diversity retention
            if random.random() < VERBOSE_KEEP_PROB:
                filtered_train_records.append(r)
                sv_fobar_kept_verbose += 1
            else:
                sv_fobar_dropped += 1

print(f"Verbose Tracks -> Kept Short: {sv_fobar_kept_short} | Kept Verbose (Diversity): {sv_fobar_kept_verbose} | Dropped: {sv_fobar_dropped}")

raw_train_records = filtered_train_records

if MAX_TRAIN_SAMPLES:
    raw_train_records = raw_train_records[:MAX_TRAIN_SAMPLES] 
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

Loading tokenizer for token-aware length gating (Patch 3 & 5)...
Starting pipeline with 95400 raw train records.
Verbose Tracks -> Kept Short: 616 | Kept Verbose (Diversity): 4010 | Dropped: 22525


In [12]:
print("Applying word-to-digit normalization and typographical cleansing...")
for rec in raw_train_records:
    q_norm = normalize_vi_math_text(rec.get('query_vi', ''))
    r_norm = normalize_vi_math_text(rec.get('response_vi', ''))
    rec['query_vi']    = clean_math_text(q_norm)
    rec['response_vi'] = clean_math_text(r_norm)

for rec in valid_records:
    q_norm = normalize_vi_math_text(rec.get('query_vi', ''))
    r_norm = normalize_vi_math_text(rec.get('response_vi', ''))
    rec['query_vi']    = clean_math_text(q_norm)
    rec['response_vi'] = clean_math_text(r_norm)

valid_normalized_count = 0
for rec in raw_train_records:
    normalized = normalize_response(rec['response_vi'])
    if normalized and is_extractable_target(normalized):
        rec['_target'] = normalized
        valid_normalized_count += 1

train_records = [r for r in raw_train_records if '_target' in r]
print(f"After normalize_response + is_extractable_target: {len(train_records)}")

initial_len = len(train_records)
train_records = [r for r in train_records if is_short_cot(r, max_steps=4)]
print(f"After is_short_cot: {len(train_records)}/{initial_len} "
      f"({len(train_records)/max(1,initial_len)*100:.1f}%)")

print("4. Deep filtering (Length, Correctness, Arithmetic)...")
train_records = [r for r in train_records if is_concise_response(r, temp_tok, max_response_tokens=MAX_RESPONSE_TOKENS)]
print(f"   -> After conciseness check: {len(train_records)}")

train_records = [r for r in train_records if is_final_answer_correct(r)]
print(f"   -> After correctness check: {len(train_records)}")

train_records = [r for r in train_records if is_arithmetically_verified(r)]
print(f"   -> After arithmetic verification: {len(train_records)}")

train_records = [r for r in train_records if is_valid_length(r, temp_tok, max_total_length=MAX_LENGTH)]
print(f"   -> After total length check: {len(train_records)}")
# Sanity check: log distribution của response lengths sau filter
resp_lens = [
    len(temp_tok(r.get("_target", ""), add_special_tokens=False)["input_ids"])
    for r in train_records
]
import numpy as np
print(f"   -> Response token stats: mean={np.mean(resp_lens):.1f}, "
      f"p50={np.percentile(resp_lens,50):.0f}, "
      f"p95={np.percentile(resp_lens,95):.0f}, "
      f"max={np.max(resp_lens)}")
del temp_tok

Applying word-to-digit normalization and typographical cleansing...
After normalize_response + is_extractable_target: 70239
After is_short_cot: 67839/70239 (96.6%)
After response token-length filter (<=150 tokens): 65868/67839 (97.1%)
4. Deep filtering (Length, Correctness, Arithmetic)...
   -> After conciseness check: 65868
   -> After correctness check: 65725
   -> After arithmetic verification: 56424
   -> After total length check: 46440
   -> Response token stats: mean=57.5, p50=53, p95=108, max=150


In [13]:
print(f"Original train subset (pre-filter): {len(raw_train_records)}")
print(f"Cleaned train (post-filter): {len(train_records)}")
print(f"Valid: {len(valid_records)}")

Original train subset (pre-filter): 72875
Cleaned train (post-filter): 46440
Valid: 500


In [ ]:
print(json.dumps(train_records[3], ensure_ascii=False, indent=2)[:2000])

In [ ]:
print(json.dumps(valid_records[2], ensure_ascii=False, indent=2)[:3000])

## SFT dataset and collator

In [14]:
class SFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length
        self.SAFE_EOS_ID = 50256

        # Pre-tokenize the anchor once for fast subsequence search
        self.stable_anchor_ids = self.tok("####", add_special_tokens=False)["input_ids"]
        self.stable_anchor_len = len(self.stable_anchor_ids)

        # Pre-build a set of token IDs that decode to math operator characters.
        # We check single-token decodings; multi-token operators are caught by context.
        self._eq_token_ids = self._find_operator_token_ids(['=', '+', '-', '*', '/'])

    def _find_operator_token_ids(self, operators: list[str]) -> set[int]:
        """
        Find all token IDs whose decoded string (stripped) is one of the operators.
        This handles cases where operators may be tokenized with surrounding spaces.
        """
        operator_ids = set()
        vocab_size = self.tok.vocab_size if hasattr(self.tok, 'vocab_size') else 50257
        # Only scan a representative range — operators are low-frequency tokens
        for token_id in range(vocab_size):
            try:
                decoded = self.tok.decode([token_id]).strip()
                if decoded in operators:
                    operator_ids.add(token_id)
            except Exception:
                continue
        return operator_ids

    def _build_arithmetic_weights(self, ids: List[int], p_ids_len: int) -> List[float]:
        n = len(ids)
        weights = [0.0] * p_ids_len + [1.0] * (n - p_ids_len)

        response_ids = ids[p_ids_len:]

        # ── Step 1: Find '=' positions and upweight computation windows ──
        WINDOW = 3  # tokens on each side of '='

        for pos, token_id in enumerate(response_ids):
            if token_id in self._eq_token_ids:
                # Upweight the window around this '='
                start = max(0, pos - WINDOW)
                end = min(len(response_ids), pos + WINDOW + 1)
                for k in range(start, end):
                    # Take the max in case windows overlap — don't double-count
                    abs_k = p_ids_len + k
                    if abs_k < n:
                        weights[abs_k] = max(weights[abs_k], COMPUTATION_WEIGHT)

        # ── Step 2: Find anchor and apply moderate anchor weight ──

        anchor_start_in_response = -1
        for j in range(len(response_ids) - self.stable_anchor_len + 1):
            if response_ids[j: j + self.stable_anchor_len] == self.stable_anchor_ids:
                anchor_start_in_response = j
                break

        if anchor_start_in_response != -1:
            # Upweight from anchor start to end of sequence
            for k in range(anchor_start_in_response, len(response_ids)):
                abs_k = p_ids_len + k
                if abs_k < n:
                    # Use max so computation window and anchor don't conflict
                    weights[abs_k] = max(weights[abs_k], ANCHOR_WEIGHT)
        else:
            # Anchor not found (shouldn't happen after filtering) — mild tail weight
            fallback_start = max(p_ids_len, n - 8)
            for k in range(fallback_start, n):
                weights[k] = max(weights[k], ANCHOR_WEIGHT)
        # ── Step 3: Hard Stop Enforcement (Critical for Inference Stability) ──
        # Đảm bảo token cuối cùng (luôn là SAFE_EOS_ID = 50256) được weight cực cao
        if ids[-1] == self.SAFE_EOS_ID:
            weights[-1] = max(weights[-1], EOS_WEIGHT)

        return weights

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())
        response = rec.get("_target", rec.get("response_vi", "")).strip()

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"]

        # 1. Dành chính xác 1 vị trí cuối cùng cho token EOS
        max_len_without_eos = self.max_length - 1
        
        ids = (p_ids + r_ids)[:max_len_without_eos]
        labels = ([-100] * len(p_ids) + r_ids)[:max_len_without_eos]

        # end bằng EOS --> giải quyết lỗi "huehuehue"
        ids.append(self.SAFE_EOS_ID)
        labels.append(self.SAFE_EOS_ID)

        # tính toán weights dựa trên mảng ids đã hoàn thiện
        # min() bảo vệ khỏi lỗi IndexError nếu prompt quá dài
        prompt_end = min(len(p_ids), len(ids) - 1) 
        weights = self._build_arithmetic_weights(ids, prompt_end)

        # safety clamp để chặn các token rác/vượt quá vocab size
        ids = [min(t, self.SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, self.SAFE_EOS_ID)) for t in labels]

        assert len(ids) == len(labels) == len(weights), "Dataset length mismatch"

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
            "loss_weights": weights,
        }

In [15]:
@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        has_weights = "loss_weights" in batch[0]
        if has_weights:
            out["loss_weights"] = []
            
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            # Right-padding (Bắt buộc khi Training Causal LM)
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            # Pad tokens không được tham gia vào hàm Loss (-100)
            out["labels"].append(x["labels"] + [-100] * pad)
            if has_weights:
                # Pad weights với 0 (sẽ bị mask anyway)
                out["loss_weights"].append(x["loss_weights"] + [0.0] * pad)
        
        result = {k: torch.tensor(v, dtype=torch.long) 
                  for k, v in out.items() if k != "loss_weights"}
        if has_weights:
            result["loss_weights"] = torch.tensor(
                out["loss_weights"], dtype=torch.float32
            )
        return result

## Greedy generation

In [16]:
# ============================================================
# 4. Stabilized Self-Consistency Generation
# ============================================================
from collections import Counter
from torch.utils.data import DataLoader
import time

def extract_numeric_answer_for_voting(text: str) -> str:
    # Captures all variations found in the data distribution
    match = re.search(r"(?i)(?:####\s*|đáp án là\s*[:：]?\s*|câu trả lời là\s*[:：]?\s*|\\boxed\{)+(-?\d+(?:[.,]\d+)?)", text)
    if match:
        return match.group(1).replace(',', '.')
    numbers = re.findall(r"(-?\d+(?:[.,]\d+)?)", text)
    if numbers:
        return numbers[-1].replace(',', '.')
    return "NONE"

In [17]:
import time
from tqdm.auto import tqdm
from collections import Counter
import torch
import json

def generate_outputs(
    model, tokenizer, records: list, 
    batch_size: int = 8,       
    max_new_tokens: int = 100,
    # --- Beam Search mode ---
    use_beam_search: bool = True,
    num_beams: int = 5,
    # --- Self-consistency mode (fallback) ---
    num_samples: int = 9,
    temperature: float = 0.4,
    top_k: int = 40,
    top_p: float = 0.90,
    # --- Shared ---
    device: str = "cuda",
    max_runtime_seconds: int = 9000,
) -> list:
    """
    Dual-mode inference:
    - use_beam_search=True : beam search (num_beams), chọn 1 best sequence
    - use_beam_search=False: self-consistency sampling (num_samples), majority voting
    """
    
    tokenizer.padding_side = "left"
    tokenizer.pad_token_id = SAFE_EOS_ID
    
    outputs = []
    start_time = time.time()
    
    # Trackers for dynamic fallback
    current_k = num_samples
    is_greedy_fallback = False
    
    for i in tqdm(range(0, len(records), batch_size), desc=f"Gen (T={temperature}, p={top_p}, k={top_k}, n={num_samples})"):
        # --- Time Budget Check ---
        elapsed = time.time() - start_time
        if elapsed > max_runtime_seconds and not is_greedy_fallback:
            print(f"\n[WARNING] Approaching time limit ({elapsed/60:.1f}m). Switching to Greedy Fallback!")
            current_k = 1
            is_greedy_fallback = True
            
        batch_records = records[i:i + batch_size]
        prompts = [PROMPT_TEMPLATE.format(q=r["query_vi"].strip()) for r in batch_records]
        
        enc = tokenizer(
            prompts, return_tensors="pt", truncation=True, padding=True,
            max_length=MAX_LENGTH, add_special_tokens=False
        ).to(device)
        
        ids = enc["input_ids"].clamp(max=model.config.vocab_size - 1)
        attention_mask = enc["attention_mask"]
        prompt_len = ids.shape[1]


        with torch.inference_mode():
            with torch.inference_mode():
                if is_greedy_fallback:
                    # Emergency: pure greedy, 1 sequence per prompt
                    gen = model.generate(
                        input_ids=ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        num_beams=1,
                        pad_token_id=SAFE_EOS_ID,
                        eos_token_id=SAFE_EOS_ID,
                    )
                    n_ret = 1
    
                elif use_beam_search:
                    # PRIMARY MODE: beam search
                    gen = model.generate(
                        input_ids=ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        num_beams=num_beams,
                        num_return_sequences=1,       # Chỉ cần best beam
                        length_penalty=0.8,           # Khuyến khích output ngắn
                        no_repeat_ngram_size=3,        # An toàn hơn repetition_penalty với math
                        early_stopping=True,
                        pad_token_id=SAFE_EOS_ID,
                        eos_token_id=SAFE_EOS_ID,
                    )
                    n_ret = 1
    
                else:
                    # self-consistency sampling
                    gen = model.generate(
                        input_ids=ids,
                        attention_mask=attention_mask,
                        max_new_tokens=max_new_tokens,
                        do_sample=True,
                        temperature=temperature,
                        top_p=top_p,
                        top_k=top_k,
                        num_return_sequences=num_samples,
                        pad_token_id=SAFE_EOS_ID,
                        eos_token_id=SAFE_EOS_ID,
                    )
                    n_ret = num_samples
        
        for j, rec in enumerate(batch_records):
            start_idx = j * n_ret
            end_idx = start_idx + n_ret
            candidates_tokens = gen[start_idx:end_idx, prompt_len:]
            candidates = tokenizer.batch_decode(candidates_tokens, skip_special_tokens=True)

            if n_ret == 1:
                # Beam search / greedy: chỉ có 1 candidate
                best_output = candidates[0]
            else:
                # Self-consistency: majority voting
                parsed = []
                for c in candidates:
                    ans_str = extract_numeric_answer_for_voting(c)
                    if ans_str != "NONE":
                        num = parse_number(ans_str)
                        if num is not None:
                            parsed.append((num, c))

                if not parsed:
                    best_output = sorted(candidates, key=len)[0]
                else:
                    counter = Counter(round(n, 4) for n, _ in parsed)
                    consensus = counter.most_common(1)[0][0]
                    consensus_texts = [c for n, c in parsed if round(n, 4) == consensus]
                    best_output = sorted(consensus_texts, key=len)[0]

            best_output = clean_model_output(best_output)
            outputs.append({
                "id": rec.get("id", len(outputs)),
                "query_vi": rec["query_vi"],
                "type": rec.get("type"),
                "model_output": best_output,
            })

    return outputs

## Training

In [18]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, SequentialSampler

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss_weights = inputs.pop("loss_weights", None)
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct     = torch.nn.CrossEntropyLoss(reduction="none")
        
        per_token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        ).view(shift_labels.shape)
        
        mask = (shift_labels != -100).float()
        
        if loss_weights is not None:
            shift_weights = loss_weights[:, 1:].contiguous().to(per_token_loss.device)
            per_token_loss = per_token_loss * shift_weights
            # FIX: Normalize by the sum of the active weights, not just the mask
            weight_sum = (mask * shift_weights).sum().clamp(min=1.0)
            weighted_loss = (per_token_loss * mask).sum() / weight_sum
        else:
            weighted_loss = (per_token_loss * mask).sum() / mask.sum().clamp(min=1.0)
            
        return (weighted_loss, outputs) if return_outputs else weighted_loss
        
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

# model.gradient_checkpointing_enable()
model.config.use_cache = False

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
train_ds = SFTDataset(train_records, tokenizer, MAX_LENGTH)
valid_ds = SFTDataset(valid_records, tokenizer, MAX_LENGTH)
collator = PadCollator(pad_id=SAFE_EOS_ID)

eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
print(
    f"per_device_bs={PER_DEVICE_BATCH_SIZE} | grad_accum={GRAD_ACCUM} | "
    f"gpus={torch.cuda.device_count()} | effective_batch={eff_batch} | "
    f"steps/epoch={steps_per_epoch}"
)

per_device_bs=8 | grad_accum=4 | gpus=2 | effective_batch=64 | steps/epoch=726


In [20]:
sample = train_ds[0]
weights = sample["loss_weights"]
ids = sample["input_ids"]

upweighted_indices = [i for i, w in enumerate(weights) if w > 1.0]
assert len(upweighted_indices) > 0, "FATAL: Anchor matching failed. Weights are 1.0 everywhere."

# Print what is actually being upweighted to ensure it caught the numbers
print("Upweighted text:", tokenizer.decode([ids[i] for i in upweighted_indices]))

Upweighted text:  đang mời 2/3 số khách tức là 84 * 2/3 = 56 khách. 84 + 56 = 140 khách. dùng là 140 + 10 = 150 đĩa. thiết là 150 * 8 = 1200 ngọn măng####đáp án là: 1200hue


In [21]:
ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    # warmup_steps=15,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    fp16=True, 
    optim="adamw_torch_fused",                   
    logging_steps=100,
    logging_strategy="steps",       
    log_level="info",                
    report_to="none",               
    save_strategy="epoch",
    save_total_limit=EPOCHS,       # keep all epoch checkpoints for manual selection
    seed=SEED,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,                  
    remove_unused_columns=False,
    group_by_length=True,                # Nhóm các chuỗi có độ dài tương đương vào cùng một batch.
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = "epoch"
else:
    ta_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
)

TRAIN_START = time.time()
trainer.train()
train_dt = time.time() - TRAIN_START
train_elapsed_min = train_dt / 60
SKIP_DECODING_SWEEP = train_elapsed_min > 120
if SKIP_DECODING_SWEEP:
    print(f"WARNING: Training took {train_elapsed_min:.1f}min. Skipping decoding sweep.")
print(f"\n[train] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model_hash = sha256_dir(OUTPUT_DIR)
with (OUTPUT_DIR / "model_hash.txt").open("w", encoding="utf-8") as f:
    f.write(model_hash + "\n")

print("Saved checkpoint to:", OUTPUT_DIR)
print("Model SHA256:", model_hash)

# Free training objects before inference reload.
del trainer, model
torch.cuda.empty_cache()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
***** Running training *****
  Num examples = 46,440
  Num Epochs = 5
  Instantaneous batch size per device = 8
  Training with DataParallel so batch size has been adjusted to: 16
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 4
  Total optimization steps = 3,630
  Number of trainable parameters = 124,439,808


Epoch,Training Loss,Validation Loss
1,3.750549,2.645073
2,2.757933,2.236896
3,1.968709,2.066456
4,1.066384,2.019202
5,0.595929,2.374789



***** Running Evaluation *****
  Num examples = 500
  Batch size = 32
Saving model checkpoint to /kaggle/working/gpt2_math_ckpt/checkpoint-726
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-726/config.json
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-726/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/gpt2_math_ckpt/checkpoint-726/model.safetensors
Deleting older checkpoint [/kaggle/working/gpt2_math_ckpt/checkpoint-819] due to args.save_total_limit

***** Running Evaluation *****
  Num examples = 500
  Batch size = 32
Saving model checkpoint to /kaggle/working/gpt2_math_ckpt/checkpoint-1452
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-1452/config.json
Configuration saved in /kaggle/working/gpt2_math_ckpt/checkpoint-1452/generation_config.json


...

Model weights saved in /kaggle/working/gpt2_math_ckpt/checkpoint-3630/model.safetensors
Deleting older checkpoint [/kaggle/working/gpt2_math_ckpt/checkpoint-2904] due to args.save_total_limit


Training completed. Do not forget to share your model on huggingface.co/models =)


Saving model checkpoint to /kaggle/working/gpt2_math_ckpt
Configuration saved in /kaggle/working/gpt2_math_ckpt/config.json
Configuration saved in /kaggle/working/gpt2_math_ckpt/generation_config.json



[train] wall time: 5740.2s (95.67 min)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/gpt2_math_ckpt/model.safetensors
tokenizer config file saved in /kaggle/working/gpt2_math_ckpt/tokenizer_config.json


Saved checkpoint to: /kaggle/working/gpt2_math_ckpt
Model SHA256: 4384bf9da9c2e96401af6f490798077f3d2e00594904a45201e9586987692346


## Inference on validation set

In [23]:
# ============================================================
# Epoch checkpoint selection + decoding sweep + final artifacts
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_TIMESTAMP = time.strftime("%Y-%m-%dT%H:%M:%S")


def checkpoint_sort_key(path: Path) -> int:
    m = re.search(r"checkpoint-(\d+)$", path.name)
    return int(m.group(1)) if m else 0


def load_inference_model(model_path: str | Path):
    print("Loading inference checkpoint:", model_path)
    m = AutoModelForCausalLM.from_pretrained(str(model_path), local_files_only=True)
    m.config.pad_token_id = SAFE_EOS_ID
    m.config.eos_token_id = SAFE_EOS_ID
    m.to(DEVICE)
    m.eval()
    return m


def decode_kwargs_from_config(config: dict) -> dict:
    if config.get("sc"):
        return dict(
            use_beam_search=False,
            num_samples=int(config.get("n", SC_NUM_SAMPLES)),
            temperature=float(config.get("temp", SC_TEMPERATURE)),
            top_k=int(config.get("top_k", SC_TOP_K)),
            top_p=float(config.get("top_p", SC_TOP_P)),
            num_beams=1,
        )
    return dict(
        use_beam_search=bool(config.get("use_beam", True)),
        num_beams=int(config.get("num_beams", NUM_BEAMS)),
        num_samples=1,
        temperature=SC_TEMPERATURE,
        top_k=SC_TOP_K,
        top_p=SC_TOP_P,
    )


def eval_loaded_model(model, records: list[dict], config: dict, label: str) -> tuple[dict, list[dict]]:
    kwargs = decode_kwargs_from_config(config)
    print(f"\n[eval:{label}] config={config}")
    outputs = generate_outputs(
        model=model,
        tokenizer=tokenizer,
        records=records,
        batch_size=8,
        max_new_tokens=MAX_NEW_TOKENS,
        device=DEVICE,
        **kwargs,
    )
    outputs = clean_output_records(outputs)
    result = evaluate(outputs, records)
    summary = result["summary"]
    print(
        f"[eval:{label}] raw={summary['raw_score']}/{summary['max_raw_score']} "
        f"exact={summary['exact_count']} extractable={summary['extractable']}/{summary['n']} "
        f"median_re={summary.get('median_rel_error')}"
    )
    return result, outputs


def summary_for_hpo(result: dict) -> dict:
    s = result["summary"]
    return {
        "raw_score": int(s["raw_score"]),
        "max_raw_score": int(s["max_raw_score"]),
        "score_10": float(s["score_10"]),
        "exact_count": int(s.get("exact_count", s["buckets"].get("10", 0))),
        "extractable_count": int(s.get("extractable", 0)),
        "numeric_pair_count": int(s.get("numeric_pairs", 0)),
        "buckets": s.get("buckets", {}),
        "score_by_type": s.get("score_by_type", {}),
        "median_rel_error": s.get("median_rel_error"),
        "capped_mean_rel_error": s.get("capped_mean_rel_error"),
    }


def save_clean_test_predictions(outputs: list[dict], path: str | Path) -> None:
    safe_outputs = []
    for i, rec in enumerate(clean_output_records(outputs)):
        safe_outputs.append({
            "id": rec.get("id", i),
            "query_vi": rec.get("query_vi", ""),
            "type": rec.get("type"),
            "model_output": rec.get("model_output", ""),
        })
    with Path(path).open("w", encoding="utf-8") as f:
        json.dump(safe_outputs, f, ensure_ascii=False, indent=2)
    print(f"Wrote {path} ({len(safe_outputs)} records)")


hpo_report = {
    "run_timestamp": RUN_TIMESTAMP,
    "training_config": {
        "epochs": EPOCHS,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "max_length": MAX_LENGTH,
        "batch_size": PER_DEVICE_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "weight_decay": WEIGHT_DECAY,
        "computation_weight": COMPUTATION_WEIGHT,
        "anchor_weight": ANCHOR_WEIGHT,
        "seed": SEED,
        "training_records": len(train_records),
        "training_minutes": float(train_dt / 60) if "train_dt" in globals() else None,
    },
    "epoch_scores": [],
    "best_epoch": None,
    "decoding_sweep": [],
    "best_decoding_config": None,
    "final_valid_score": None,
    "type_routing_used": bool(TYPE_ROUTING_USED),
}

# Baseline evaluation is kept lightweight and written to required artifact paths.
print("\n=== Baseline validation evaluation from base model ===")
baseline_model = load_inference_model(MODEL_NAME)
baseline_config = dict(use_beam=USE_BEAM_SEARCH, num_beams=NUM_BEAMS, sc=not USE_BEAM_SEARCH)
baseline_result, baseline_outputs = eval_loaded_model(baseline_model, valid_records, baseline_config, "baseline")
baseline_outputs = save_prediction_file(baseline_outputs, BASELINE_OUTPUT_PATH)
baseline_result = save_evaluation_report(BASELINE_OUTPUT_PATH, valid_records, BASELINE_REPORT_PATH)
del baseline_model
torch.cuda.empty_cache()

ckpt_dirs = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=checkpoint_sort_key)
if not ckpt_dirs:
    print("No epoch checkpoint dirs found; falling back to OUTPUT_DIR.")
    ckpt_dirs = [OUTPUT_DIR]

print("\n=== Epoch checkpoint evaluation ===")
default_config = dict(use_beam=USE_BEAM_SEARCH, num_beams=NUM_BEAMS, sc=not USE_BEAM_SEARCH)
for epoch_idx, ckpt in enumerate(ckpt_dirs, start=1):
    model_ckpt = load_inference_model(ckpt)
    epoch_result, _ = eval_loaded_model(model_ckpt, valid_records, default_config, f"epoch{epoch_idx}")
    epoch_summary = summary_for_hpo(epoch_result)
    hpo_report["epoch_scores"].append({
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),
        "raw_score": epoch_summary["raw_score"],
        "exact_count": epoch_summary["exact_count"],
        "extractable_count": epoch_summary["extractable_count"],
        "score_by_type": epoch_summary["score_by_type"],
        "median_rel_error": epoch_summary["median_rel_error"],
        "capped_mean_rel_error": epoch_summary["capped_mean_rel_error"],
    })
    del model_ckpt
    torch.cuda.empty_cache()

best_epoch_entry = max(
    hpo_report["epoch_scores"],
    key=lambda x: (x["raw_score"], x["exact_count"], x["extractable_count"]),
)
best_ckpt = Path(best_epoch_entry["checkpoint"])
hpo_report["best_epoch"] = int(best_epoch_entry["epoch"])
print("\nEpoch scores:")
for row in hpo_report["epoch_scores"]:
    print(f"Epoch {row['epoch']}: {Path(row['checkpoint']).name} raw_score={row['raw_score']} exact={row['exact_count']}")
print("Best epoch:", hpo_report["best_epoch"], "->", best_ckpt)

# Decoding sweep on best checkpoint.
best_decoding_label = "sc9_t04"
best_decoding_config = dict(use_beam=False, sc=True, n=9, temp=0.4, top_k=40, top_p=0.90)
best_sweep_outputs = None
best_sweep_result = None

if SKIP_DECODING_SWEEP:
    print("\nSkipping decoding sweep due runtime guard; using sc9_t04.")
else:
    print("\n=== Decoding sweep on best checkpoint ===")
    sweep_model = load_inference_model(best_ckpt)
    best_sweep_key = None
    for label, config in DECODING_GRID:
        sweep_result, sweep_outputs = eval_loaded_model(sweep_model, valid_records, config, label)
        sweep_summary = summary_for_hpo(sweep_result)
        hpo_report["decoding_sweep"].append({
            "label": label,
            "config": config,
            **sweep_summary,
        })
        key = (sweep_summary["raw_score"], sweep_summary["exact_count"], sweep_summary["extractable_count"])
        if best_sweep_key is None or key > best_sweep_key:
            best_sweep_key = key
            best_decoding_label = label
            best_decoding_config = config
            best_sweep_outputs = sweep_outputs
            best_sweep_result = sweep_result
    del sweep_model
    torch.cuda.empty_cache()

hpo_report["best_decoding_config"] = best_decoding_label
print("Best decoding config:", best_decoding_label, best_decoding_config)

# Final validation output/report uses the best checkpoint + best decoding config.
if best_sweep_outputs is None or best_sweep_result is None:
    final_model = load_inference_model(best_ckpt)
    valid_result, valid_outputs = eval_loaded_model(final_model, valid_records, best_decoding_config, "final_valid")
    del final_model
    torch.cuda.empty_cache()
else:
    valid_outputs = best_sweep_outputs
    valid_result = best_sweep_result

valid_outputs = save_prediction_file(valid_outputs, VALID_OUTPUT_PATH)
valid_result = save_evaluation_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
summary = valid_result["summary"]
hpo_report["final_valid_score"] = int(summary["raw_score"])

print("\nFinal validation score:")
print(f'{summary["raw_score"]} / {summary["max_raw_score"]}  ({summary["score_pct"]*100:.2f}%)')
print(f'Score /10: {summary["score_10"]:.2f}')
print("Buckets:", summary["buckets"])
print("Score by type:", json.dumps(summary.get("score_by_type", {}), ensure_ascii=False, indent=2)[:4000])

# Phase-2-safe test prediction block. If test.json is absent during Phase 1, write [] so artifact checks pass.
if TEST_FILE.exists():
    print("\n=== Test prediction ===")
    test_records = load_records(TEST_FILE)
    for rec in test_records:
        q_norm = normalize_vi_math_text(rec.get("query_vi", ""))
        rec["query_vi"] = clean_math_text(q_norm)
    test_model = load_inference_model(best_ckpt)
    test_outputs = generate_outputs(
        model=test_model,
        tokenizer=tokenizer,
        records=test_records,
        batch_size=8,
        max_new_tokens=MAX_NEW_TOKENS,
        device=DEVICE,
        **decode_kwargs_from_config(best_decoding_config),
    )
    del test_model
    torch.cuda.empty_cache()
    save_clean_test_predictions(test_outputs, TEST_OUTPUT_PATH)
else:
    print("test.json not found yet; writing empty Phase 1 placeholder test_predictions.json.")
    save_clean_test_predictions([], TEST_OUTPUT_PATH)

experiment_report = {
    "run_id": "rewind3_final_stretch_epoch_sweep",
    "base_notebook": "rewind3.ipynb",
    "model_name": MODEL_NAME,
    "data_dir": str(DATA_DIR),
    "prompt_template": PROMPT_TEMPLATE,
    "safe_eos_id": SAFE_EOS_ID,
    "training_config": hpo_report["training_config"],
    "best_epoch": hpo_report["best_epoch"],
    "best_checkpoint": str(best_ckpt),
    "best_decoding_config": best_decoding_label,
    "best_decoding_params": best_decoding_config,
    "final_validation_summary": valid_result["summary"],
    "baseline_summary": baseline_result["summary"],
    "type_routing_used": bool(TYPE_ROUTING_USED),
}

with EXPERIMENT_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(experiment_report, f, ensure_ascii=False, indent=2)

with HPO_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(hpo_report, f, ensure_ascii=False, indent=2)

print("Wrote", EXPERIMENT_REPORT_PATH)
print("Wrote", HPO_REPORT_PATH)


loading configuration file /kaggle/working/gpt2_math_ckpt/config.json
Model config GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": 50256,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.0,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version":

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

loading configuration file /kaggle/working/gpt2_math_ckpt/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 50256,
  "eos_token_id": 50256,
  "use_cache": true
}



GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

##### Beam search vs Self consistency

In [24]:
print("Quick A/B cell skipped: replaced by epoch checkpoint evaluation and decoding sweep above.")


=== A/B TEST: Beam Search vs Self-Consistency ===

[A] Beam Search (num_beams=5)...


Gen (T=0.4, p=0.9, k=40, n=9):   0%|          | 0/10 [00:00<?, ?it/s]

Beam: score=27.00% | bucket10=20 | extractable=79/80

[B] Self-Consistency (K=9, T=0.4)...


Gen (T=0.4, p=0.9, k=40, n=9):   0%|          | 0/10 [00:00<?, ?it/s]

SC:   score=28.88% | bucket10=22 | extractable=77/80

[DECISION] Self-Consistency wins -> dùng cho full inference


In [26]:
print("Full validation generation already completed by final stretch cell.")



Running full inference with SELF-CONSISTENCY...


Gen (T=0.4, p=0.9, k=40, n=9):   0%|          | 0/63 [00:00<?, ?it/s]


Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi 1 trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước 8 ô, ở lượt thứ hai, cô ấy di chuyển 2 ô nhưng bị đẩy lùi lại 5 ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước 6 ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "2 ô, nhưng bị lùi lại 5 ô, do đó cô ấy di chuyển 2 - 5 = -3 ô. 48 - (-3) = 57 ô. 57 - 2 = 57 ô\n####đáp án là: 57huehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehue"
}


In [ ]:
# import itertools
# import pandas as pd

# def find_best_hyperparameters(model, tokenizer, valid_records, subset_size=64):
#     print(f"=== KHỞI ĐỘNG GRID SEARCH TRÊN {subset_size} SAMPLES ===")
    
#     random.seed(42)
#     sweep_records = random.sample(valid_records, min(subset_size, len(valid_records)))

#     param_grid = {
#         "num_samples": [5, 7, 9, 12],
#         "temperature": [0.3, 0.4, 0.5,0.6, 0.7],
#         "top_k": [30, 40, 50],
#         "top_p": [0.85, 0.90, 0.92]
#     }
    
#     keys = param_grid.keys()
#     combinations = [dict(zip(keys, v)) for v in itertools.product(*param_grid.values())]
    
#     results = []
    
#     for idx, kwargs in enumerate(combinations):
#         print(f"\n[Config {idx+1}/{len(combinations)}] Testing: {kwargs}")
        
#         # Tạo output
#         sweep_outputs = generate_outputs(
#             model=model,
#             tokenizer=tokenizer,
#             records=sweep_records,
#             batch_size=8,
#             max_new_tokens=150,
#             **kwargs
#         )
        
#         # Đánh giá bằng hàm evaluate có sẵn của bạn
#         eval_result = evaluate(sweep_outputs, sweep_records)
#         summary = eval_result["summary"]
        
#         score_10_pct = summary["buckets"]["10"] / summary["n"]
#         raw_score = summary["raw_score"]
        
#         print(f"-> Score 10s: {summary['buckets']['10']}/{summary['n']} | Raw Score: {raw_score}")
        
#         results.append({
#             **kwargs,
#             "score_10_count": summary["buckets"]["10"],
#             "raw_score": raw_score,
#             "extractable": summary["extractable"],
#             "score_pct": summary["score_pct"]
#         })
        
#     # 3. Phân tích kết quả
#     df_results = pd.DataFrame(results)
#     df_results = df_results.sort_values(by=["score_10_count", "raw_score", "extractable"], ascending=[False, False, False])
    
#     print("\n" + "="*80)
#     print(" BẢNG XẾP HẠNG HYPERPARAMETERS (TỐI ƯU CHO SCORE 10)")
#     print("="*80)
#     print(df_results.to_markdown(index=False))
    
#     best_params = df_results.iloc[0].to_dict()
#     # Loại bỏ các key kết quả để trả về đúng dict param
#     best_params = {k: best_params[k] for k in param_grid.keys()}
    
#     print(f"\n[!] BỘ THAM SỐ CHIẾN THẮNG: {best_params}")
#     return best_params

# inference_model = AutoModelForCausalLM.from_pretrained(
#     OUTPUT_DIR,
#     local_files_only=True
# )
# inference_model.to("cuda")
# inference_model.eval()

# best_params = find_best_hyperparameters(inference_model, tokenizer, valid_records, subset_size=60)

In [ ]:
# final_best_params = {
#     "num_samples": int(best_params["num_samples"]),
#     "temperature": float(best_params["temperature"]),
#     "top_k": int(best_params["top_k"]),
#     "top_p": float(best_params["top_p"])
# }

# print(f"\nRunning full inference with {'BEAM SEARCH' if FINAL_USE_BEAM else 'SELF-CONSISTENCY'}...")
# valid_outputs = generate_outputs(
#     model=inference_model,
#     tokenizer=tokenizer,
#     records=valid_records,
#     batch_size=8,
#     max_new_tokens=MAX_NEW_TOKENS,
#     use_beam_search=FINAL_USE_BEAM,
#     # Beam params
#     num_beams=NUM_BEAMS,
#     num_samples=SC_NUM_SAMPLES,
#     temperature=SC_TEMPERATURE,
#     top_k=SC_TOP_K,
#     top_p=SC_TOP_P,
# )

# print("\nExample output:")
# print(json.dumps(valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

## Evaluate

In [27]:
print("Validation output/report already written by final stretch cell.")



Example output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi 1 trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước 8 ô, ở lượt thứ hai, cô ấy di chuyển 2 ô nhưng bị đẩy lùi lại 5 ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước 6 ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "model_output": "2 ô, nhưng bị lùi lại 5 ô, do đó cô ấy di chuyển 2 - 5 = -3 ô. 48 - (-3) = 57 ô. 57 - 2 = 57 ô\n####đáp án là: 57huehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehuehue"
}
{
  "n": 500,
  "raw_score": 1540,
  "max_raw_score": 5000,
  "score_10": 3.08,
  "score_pct": 0.308,
  "extractable": 494,
  "numeric_pairs": 488,
  "buckets": {
    "10": 143,
    "5": 6,
    "1": 80,
    "0": 271
  },
  "rel_error_mean": 57332.80627531676
}
Wrote /kaggle/working/valid_report.json

Final validation score:
1540 / 5000  (30.80%

In [28]:
# ============================================================
# 10. Detailed Error analysis
# ============================================================
rows = valid_result["rows"]
bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
good = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 10]

print("Good:", len(good), "| Bad:", len(bad))

for i, r in bad[:2]:
    pred = valid_outputs[i]
    gold = valid_records[i]
    print("=" * 100)
    print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
    print("QUERY:", gold["query_vi"][:500])
    print("\nGOLD:", gold["response_vi"][-500:])
    print("\nPRED:", pred["model_output"][:1000])

Good: 143 | Bad: 271
IDX: 0 | type: GSM_Rephrased | rel_error: 0.5405405405405406
QUERY: Nếu Susan đang chơi 1 trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước 8 ô, ở lượt thứ hai, cô ấy di chuyển 2 ô nhưng bị đẩy lùi lại 5 ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước 6 ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?

GOLD: Ở lượt đầu tiên, Susan tiến về phía trước 8 ô, vì vậy cô ấy hiện ở ô 8. Ở lượt thứ hai, cô ấy tiến về phía trước 2 ô, nhưng bị đẩy lùi 5 ô, vì vậy cô ấy kết thúc ở ô 8 + 2 - 5 = 5 Ở lượt thứ 3, cô ấy tiến về phía trước 6 ô nên lúc này cô ấy đang ở ô 5 + 6 = 11. Ô cuối cùng thắng là 48 nên Susan cần di chuyển thêm 48 - 11 = 37 ô nữa để đến ô cuối cùng và giành chiến thắng tro choi. #### 37 Đáp án là: 37

PRED: 2 ô, nhưng bị lùi lại 5 ô, do đó cô ấy di chuyển 2 - 5 = -3 ô. 48 - (-3) = 57 ô. 57 - 2 = 57 ô
####đáp án là: 57huehuehuehuehuehuehuehuehuehuehuehueh

In [29]:
from collections import Counter, defaultdict
import pandas as pd

class VietnameseMathErrorAnalyzer:
    def __init__(self, valid_report_path: str | Path, valid_output_path: str | Path, tokenizer=None):
        self.report_path = Path(valid_report_path)
        self.output_path = Path(valid_output_path)
        self.tokenizer = tokenizer
        
        # Load datasets
        with self.report_path.open("r", encoding="utf-8") as f:
            self.report_data = json.load(f)
        with self.output_path.open("r", encoding="utf-8") as f:
            self.output_data = json.load(f)
            
        self.rows = self.report_data["rows"]
        self.summary = self.report_data["summary"]
        
        # Build faster lookup mapping
        self.pred_map = {i: x for i, x in enumerate(self.output_data)}
        self.df = None

    # ============================================================
    # 1. Failure Taxonomy & GPT2 Degeneration Core
    # ============================================================
    def _classify_failure_modes(self, row: dict, pred_item: dict) -> list[str]:
        """Heuristic rule engine classifying classic decoder-only math degradation signatures."""
        modes = []
        output_text = pred_item.get("model_output", "")
        gold_text = row.get("gold_answer", "") or ""
        
        # Handle cases where answer tracking fails completely
        if not row["extractable"]:
            modes.append("extraction_failure")
            
        # Detect Premature End of Sequence (EOS) vs Verbosity Truncation
        approx_words = len(output_text.split())
        if approx_words <= 4 and row["score"] == 0:
            modes.append("premature_eos")
        elif approx_words >= 195:  # Close to the 200 token MAX_NEW_TOKENS boundary
            modes.append("truncation_failure")

        # Repetition Loop Detectors (Classic GPT-2 Signature)
        # Check for repeated N-grams or continuous phrases
        words = output_text.lower().split()
        if len(words) > 10:
            trigrams = [" ".join(words[i:i+3]) for i in range(len(words)-2)]
            if len(trigrams) > 0:
                most_common_tri = Counter(trigrams).most_common(1)[0]
                if most_common_tri[1] > 4:  # Trigram repeats more than 4 times
                    modes.append("repetition_loop")

        if row["extractable"] and row["score"] == 0:
            pred_num = row["pred_num"]
            gold_num = row["gold_num"]
            
            if pred_num is not None and gold_num is not None:
                # Sign Mistakes
                if pred_num == -gold_num and gold_num != 0:
                    modes.append("operator_sign_mistake")
                # Decimal placement shifts
                elif gold_num != 0 and pred_num != 0:
                    ratio = abs(pred_num / gold_num)
                    # Guard against float overflow/underflow resulting in inf or 0
                    if ratio > 0 and not np.isinf(ratio):
                        log_ratio = np.log10(ratio)
                        # Check if log_ratio is extremely close to an integer (e.g., 10^1, 10^-2)
                        if abs(log_ratio - round(log_ratio)) < 1e-4:
                            modes.append("decimal_placement_error")
                        else:
                            modes.append("arithmetic_mistake")
                    else:
                        modes.append("arithmetic_mistake")
                else:
                    # Catch-all for when one value is 0 but they don't match
                    modes.append("arithmetic_mistake")
            else:
                modes.append("formatting_drift")
                
        # Bilingual Leakage Heuristics
        en_words = re.findall(r"\b(the|answer|is|let|hence|therefore|step|solution)\b", output_text, re.I)
        if len(en_words) >= 2:
            modes.append("bilingual_leakage")
            
        if not modes and row["score"] < 10:
            modes.append("reasoning_collapse")
            
        return modes if modes else ["clean_pass"]

    # ============================================================
    # 2. Pipeline Execution Engine
    # ============================================================
    def build_diagnostic_dataframe(self):
        """Assembles comprehensive failure profiling metrics across columns."""
        processed_rows = []
        
        for idx, r in enumerate(self.rows):
            p = self.pred_map[idx]
            out_text = p.get("model_output", "")
            
            # Compute character lengths & math structural density
            pred_len_chars = len(out_text)
            pred_words = out_text.split()
            math_ops_count = len(re.findall(r"[\+\-\*\/=]|\\frac", out_text))
            math_density = math_ops_count / max(1, len(pred_words))
            
            # Identify precise taxonomy classes
            failure_categories = self._classify_failure_modes(r, p)
            
            # Compute raw tokenizer sequence lengths if optional tokenizer available
            token_count = len(self.tokenizer.encode(out_text)) if self.tokenizer else len(pred_words) * 1.3
            
            processed_rows.append({
                "id": r["id"],
                "type": r["type"],
                "score": r["score"],
                "extractable": int(r["extractable"]),
                "rel_error": r["rel_error"] if r["rel_error"] is not None else np.nan,
                "pred_len_tokens": token_count,
                "math_density": math_density,
                "failures": failure_categories,
                "primary_failure": failure_categories[0]
            })
            
        self.df = pd.DataFrame(processed_rows)
        return self.df

    # ============================================================
    # 3. Analytics & Score-Bucket Sweeper
    # ============================================================
    def generate_dashboard_metrics(self) -> dict:
        """Computes structural aggregations mapping failure classes directly to Score loss."""
        if self.df is None:
            self.build_diagnostic_dataframe()
            
        metrics = {}
        
        # Explode failures list to get true distribution counts
        exploded_df = self.df.explode("failures")
        failure_counts = exploded_df["failures"].value_counts()
        avg_score_by_failure = exploded_df.groupby("failures")["score"].mean()
        
        # Assemble Primary Damage Inventory Matrix
        damage_matrix = pd.DataFrame({
            "Occurrences": failure_counts,
            "Mean_Score_Impact": avg_score_by_failure
        }).sort_values(by="Occurrences", ascending=False)
        
        metrics["failure_damage_table"] = damage_matrix
        
        # Analyze performance across math problem tracks
        metrics["type_performance"] = self.df.groupby("type")["score"].agg(["count", "mean"]).to_dict(orient="index")
        
        # Metric Extraction Diagnostics Dashboard
        metrics["extraction_leakage_rate"] = 1.0 - (self.df["extractable"].sum() / len(self.df))
        
        return metrics

    # ============================================================
    # 4. Simulation Engine: Extraction Fallbacks (No Retraining ROI)
    # ============================================================
    def simulate_fallback_extractor_gain(self) -> float:
        """Simulates potential validation point recovery using post-hoc regex without retraining."""
        simulated_score = 0
        recovered_points = 0
        
        for idx, r in enumerate(self.rows):
            p = self.pred_map[idx]
            out_text = p.get("model_output", "")
            
            if r["score"] > 0:
                simulated_score += r["score"]
                continue
                
            # Simulate an aggressive reverse number matching scraper
            nums = re.findall(r"-?\d+(?:[,\.]\d+)?", out_text)
            if nums:
                fallback_pred_str = nums[-1].replace(",", ".")
                try:
                    fallback_pred = float(fallback_pred_str)
                    gold_num = float(r["gold_num"]) if r["gold_num"] is not None else None
                    
                    if gold_num is not None:
                        denom = max(1.0, abs(gold_num))
                        error = abs(fallback_pred - gold_num) / denom
                        
                        # Calculate recovery tier
                        this_score = 0
                        if error <= 0.01: this_score = 10
                        elif error <= 0.10: this_score = 5
                        elif error <= 0.50: this_score = 1
                        
                        recovered_points += this_score
                        simulated_score += this_score
                except ValueError:
                    pass
                    
        baseline_score = self.summary["raw_score"]
        max_possible = len(self.rows) * 10
        print(f"\n[EXTRACTION SIMULATOR] Baseline Validation Score: {baseline_score}/{max_possible}")
        print(f"[EXTRACTION SIMULATOR] Simulated Fallback Score: {baseline_score + recovered_points}/{max_possible}")
        print(f"[EXTRACTION SIMULATOR] Net Recovery Delta: +{recovered_points} points ({((recovered_points)/max_possible)*100:.2f}%)")
        return recovered_points

    # ============================================================
    # 5. Formatted Markdown Reporting Tool
    # ============================================================
    def print_terminal_markdown_report(self):
        """Generates clear, scannable terminal outputs for experiment tracking."""
        metrics = self.generate_dashboard_metrics()
        
        print("\n" + "="*80)
        print("         COMPETITION DATA-CENTRIC ERROR DIAGNOSTIC REPORT")
        print("="*80)
        
        print(f"\n### SUMMARY PROFILE")
        print(f"* Raw Base Validation Score: {self.summary['raw_score']} / {self.summary['max_raw_score']} ({self.summary['score_pct']*100:.2f}%)")
        print(f"* Total Dataset Samples Analyzed: {self.summary['n']}")
        print(f"* Unextractable Pipeline Leakage: {metrics['extraction_leakage_rate']*100:.2f}%")
        
        print("\n### FAILURE MATRIX DAMAGE DISTRIBUTION")
        print(metrics["failure_damage_table"].to_markdown())
        
        print("\n### SUB-TRACK TRACKING PERFORMANCE")
        track_df = pd.DataFrame.from_dict(metrics["type_performance"], orient="index")
        print(track_df.to_markdown())
        
        self.simulate_fallback_extractor_gain()
        print("\n" + "="*80)

In [30]:
analyzer = VietnameseMathErrorAnalyzer(VALID_REPORT_PATH, VALID_OUTPUT_PATH, tokenizer=tokenizer)
analyzer.print_terminal_markdown_report()
metrics = analyzer.generate_dashboard_metrics()
error_analysis_payload = {
    "summary": analyzer.summary,
    "failure_damage_table": metrics["failure_damage_table"].reset_index().to_dict(orient="records"),
    "type_performance": metrics["type_performance"],
    "extraction_leakage_rate": metrics["extraction_leakage_rate"],
}
with ERROR_ANALYSIS_PATH.open("w", encoding="utf-8") as f:
    json.dump(error_analysis_payload, f, ensure_ascii=False, indent=2, default=str)
print("Wrote", ERROR_ANALYSIS_PATH)



         COMPETITION DATA-CENTRIC ERROR DIAGNOSTIC REPORT

### SUMMARY PROFILE
* Raw Base Validation Score: 1540 / 5000 (30.80%)
* Total Dataset Samples Analyzed: 500
* Unextractable Pipeline Leakage: 1.20%

### FAILURE MATRIX DAMAGE DISTRIBUTION
| failures                |   Occurrences |   Mean_Score_Impact |
|:------------------------|--------------:|--------------------:|
| arithmetic_mistake      |           249 |             0       |
| clean_pass              |           138 |            10       |
| reasoning_collapse      |            80 |             1.25    |
| repetition_loop         |            39 |             1.53846 |
| decimal_placement_error |             9 |             0       |
| extraction_failure      |             6 |             0       |
| formatting_drift        |             6 |             0       |
| operator_sign_mistake   |             1 |             0       |

### SUB-TRACK TRACKING PERFORMANCE
|                |   count |     mean |
|:--------------

In [ ]:
required_files = [
    VALID_OUTPUT_PATH, VALID_REPORT_PATH,
    BASELINE_OUTPUT_PATH, BASELINE_REPORT_PATH,
    TEST_OUTPUT_PATH, EXPERIMENT_REPORT_PATH,
    ERROR_ANALYSIS_PATH, HPO_REPORT_PATH,
]
missing = [str(f) for f in required_files if not f.exists()]
if not OUTPUT_DIR.exists():
    missing.append(str(OUTPUT_DIR))
if missing:
    raise RuntimeError(f"MISSING REQUIRED OUTPUT FILES: {missing}")

with open(TEST_OUTPUT_PATH, encoding="utf-8") as f:
    test_preds = json.load(f)
for rec in test_preds:
    assert "response_vi" not in rec, "Gold answer leaked into test predictions!"
    assert "gold_answer" not in rec, "Gold answer leaked into test predictions!"
    assert "pred_answer" not in rec, "Validation field leaked into test predictions!"
    assert "model_output" in rec, "Missing model_output in test predictions!"

print(f"All {len(required_files)} required files present.")
print(f"test_predictions.json: {len(test_preds)} records, no gold answer leakage.")

print("\nKaggle output folder:")
for p in sorted(Path("/kaggle/working").glob("*")):
    print("-", p)
